In [1]:
import evaluation_main
import evaluation_utils
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import fitz

In [ ]:
#documet examples
#doc_id = "2023_73325_200"
#doc_id = "2023_73413_200"
#doc_id = "2010_923067_200"

#evaluation_utils.test_and_visualize_doc(doc_id)

In [2]:
folder_path = '../valideringssett/dokumenter/'

total_results, total_tp, total_fp, total_fn = evaluation_main.evaluate_model(folder_path)

['230685 45633']
['42 114 184 815']
[]
[]
2013_405295_200
['d\nse Husbanken Pantedokument  fast eiendom\n\nDette pantedokumentet kan ikke overdras\n\n Saksnr\n11523781\nDepotnrDoknr\n Rekvirentens navn returadresse fdselsorgnr\nAuTVvIK As\nPostpoks 75 Doknr 405295 Tinglyst 22052013\nYRAIG AVALDSNES STATENS KARTVERK FAST EIENDOM\n13 039 3i3 Tinglysingsgjenpart\nPantsettere\nFdselsnrOrganisasjonsnr  Navn\n230685 45633 Jeanette Braut Kallevik\nAngivelse av pantekravets strrelse\nc Belp Belp med bokstaver Valuta\n1516 000 enmillionfemhundreogsekstentusenkroner\n Panthaver Orgnr\n\n Husbanken 942 114 184\n Pantobjekt\n6  Kommunenr  Kommunens navn  Gnr  Bnr  Festenr  Seksinr  Ideell andel\n 1149 Karmy kommune 149 4Y50\nE\n2\nF\noa\nrd\n\nAvtalt prioritet\n\nForbud mot visse rettslige disposisjoner\n\nEtterstende vilkr skal ikke tinglyses\nGjeldsansvar\n\nPantsetter erkjenner herved  skylde Husbanken kr 1516 000\n\nDet nrmere forhold mellom pantsetter og panthaver herunder om forfall av det p

KeyboardInterrupt: 

## RESULTS ALL DOCUMENTS

Total TP: 686, Total FP: 9, Total FN: 823

In [ ]:

def get_metrics_and_cm(total_tp, total_fp, total_fn):

    precision = total_tp / (total_tp + total_fp)
    recall = total_tp / (total_tp + total_fn)
    f1 = 2 * (precision * recall) / (precision + recall)

    print('Precision:', precision)
    print('Recall:', recall)
    print('F1:', f1)
    print('Accuracy:', total_tp/(total_tp + total_fp + total_fn))

    # Define the confusion matrix
    conf_matrix = np.array([[total_tp, total_fp], 
                            [total_fn, 0]])

    # Labels for each cell
    group_names = ['True Positive', 'False Positive', 'False Negative','True Negative']
    group_counts = ["{0:0.0f}".format(value) for value in conf_matrix.flatten()]
    group_percentages = ["{0:.2%}".format(value) for value in conf_matrix.flatten() / np.sum(conf_matrix)]
    labels = (np.asarray(["{}\n{}\n{}".format(name, count, pct) for name, count, pct in zip(group_names, group_counts, group_percentages)])).reshape(2,2)

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(conf_matrix, interpolation='nearest', cmap='Blues')

    # We want to show all ticks...
    ax.set(xticks=np.arange(conf_matrix.shape[1]),
        yticks=np.arange(conf_matrix.shape[0]),
        xticklabels=['Positives','Negatives'], 
        yticklabels=['Positives','Negatives'],
        title='Confusion Matrix')

    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
            rotation_mode="anchor")

    # Loop over data dimensions and create text annotations.
    for i in range(conf_matrix.shape[0]):
        for j in range(conf_matrix.shape[1]):
            ax.text(j, i, labels[i, j],
                    ha="center", va="center",
                    color="white" if conf_matrix[i, j] > conf_matrix.max() / 2 else "black")

    plt.ylabel('Predicted labels')
    plt.xlabel('Actual labels')
    plt.tight_layout()
    plt.show()


In [ ]:

get_metrics_and_cm(686, 9, 823)

In [ ]:

#Get metadata from pdf
def get_metadata(pdf_path):
    doc = fitz.open(pdf_path)
    metadata = doc.metadata
    doc.close()
    return metadata

folder_path = '../valideringssett/dokumenter'

dokument_identer_elektronisk = []

for doc in os.listdir(folder_path):
    metadata = get_metadata(os.path.join(folder_path, doc))
    if metadata['title'] == 'Dokument til signering':
        dokument_identer_elektronisk.append(doc[:-4])
        #Save a copy of the pdf file to the folder '../valideringssett/elektrnisk_tinglyst'
        os.rename(os.path.join(folder_path, doc), os.path.join('../valideringssett/elektronisk_tinglyst', doc))

In [ ]:
folder_path = '../valideringssett/elektronisk_tinglyst/'

total_results, total_tp, total_fp, total_fn = evaluation_main.evaluate_model(folder_path)

## RESULTS ELEKTRONISK TINGLYST
Total TP: 160, Total FP: 0, Total FN: 2

In [ ]:
get_metrics_and_cm(162, 0, 0)

## SAVE DOCUMENTS WITH PREDICTED SLADS

In [ ]:
folder_path = '../valideringssett/elektronisk_tinglyst/'

for doc in os.listdir(folder_path):
    os.mkdir(f'resultater_elektronisk_tinglyst/{doc[:-4]}')
    images_with_bbs = evaluation_utils.test_and_visualize_doc(doc[:-4])
    for i, image in enumerate(images_with_bbs):
        image.savefig(f'resultater_elektronisk_tinglyst/{doc[:-4]}/{i}.png')